In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [ ]:
DATA_FILE_PATH="insurance.csv"

In [ ]:
class LinearRegression:
    def __init__(self,features:np.array,batch_size=512,learning_rate=0.5,training_percentage = 0.8):
        self.coeff = np.random.random(size=(features.shape[1]-1))
        # self.inputs = np.zeros((num_features+1))
        self.batch_size:int = batch_size
        self.training_percentage = training_percentage
        self.predictions = np.array([])
        self.actual_values = np.array([])
        self.learning_rate = learning_rate
        split_index:int = int(features.shape[0]*training_percentage)
        self.X = features[:,:-1]
        self.Y = features[:,-1:]

        self.train_data,self.test_data = np.split(features,[split_index],axis=0)

        self.train_x,self.test_x,self.train_y,self.test_y = train_test_split(self.X,self.Y,test_size=0.2,random_state=42)
        self.inputs = self.train_data[:,:-1]
        self.outputs = self.train_data[:,-1:].reshape((self.train_data.shape[0]))

        
        self.cost_function_values = np.array([])
        self.train_std:float
        self.train_mean:float
        self.epochs = 1000

        self.data_ptr:int = 0
        self.batch_inputs = np.zeros((batch_size,features.shape[1]-1))
    
    def hx(self,inputs):
        """ calculates the prediction value """
        return np.matmul(inputs,self.coeff)
    
    def cost_funtion(self):
        """Calculates the Cost function using mse"""
        rmse = np.sqrt(((self.predictions-self.outputs)**2).mean())
        return rmse
    def r_squared(self):
        res = ((self.predictions - self.outputs)**2).sum() / ((self.outputs - self.outputs.mean())**2 ).sum()
        return (1 - res)
    def learn(self,batch_size):
        delta = self.predictions-self.outputs
        delta = delta.reshape((delta.shape[0],1))
        gradient = self.inputs*delta/delta.shape[0]
        self.coeff -= self.learning_rate * gradient.sum(axis=0)

    def train_model(self):
        
        for _ in range (self.epochs):
            # print(self.cost_funtion())
            # np.random.shuffle(self.train_data)
            # self.inputs = self.train_data[:,:-1]
            self.inputs = self.train_x
            self.train_mean = self.inputs.mean(axis=0)
            self.train_std = self.inputs.std(axis=0)
            self.train_std[0] = 1 # fix the bias column error
            self.inputs = (self.inputs - self.train_mean) / self.train_std
            self.inputs[:,0] = 1 # fix the bias columns going zero 
            # self.outputs = self.train_data[:,-1:].reshape((self.train_data.shape[0]))
            self.outputs = self.train_y.reshape((self.train_y.shape[0]))

            self.data_ptr = 0
            num_rows:int = self.train_x.shape[0]

            self.predictions = self.hx(inputs=self.inputs)
            self.learn(batch_size=num_rows)
            curr_cost = self.cost_funtion()
            self.cost_function_values=np.append(arr=self.cost_function_values,values=[curr_cost])

    def test_model(self):
        self.metrics = np.array([])
        for _ in range(self.epochs):
            self.data_ptr = 0
            num_rows:int = self.test_x.shape[0] 
            # np.random.shuffle(self.test_data)
            self.inputs = self.test_x
            self.inputs = (self.inputs - self.train_mean.reshape((1,self.train_mean.shape[0]))) / self.train_std
            self.inputs[:,0] = 1 # set the bias column
            self.outputs = self.test_y.reshape((self.test_data.shape[0]))

            self.predictions = self.hx(inputs=self.inputs)
            curr_cost = self.r_squared()
            self.metrics=np.append(arr=self.metrics,values=[curr_cost])
        
    def print_coeff(self):
        print(self.coeff)



In [ ]:
if __name__=='__main__':
    # preprocessin and refining
    data = pd.read_csv(DATA_FILE_PATH).drop(columns='region')
    data.loc[data['sex']=='male','sex'] = 1
    data.loc[data['sex']=='female','sex'] = 0
    data.loc[data['smoker']=='yes','smoker'] = 1
    data.loc[data['smoker']=='no','smoker'] = 0
    data.insert(loc=0,column="bias",value=1)
    features = data.to_numpy().astype(float)
    # np.random.shuffle(features)
    model = LinearRegression(features=features)
    model.train_model()
    
    

In [ ]:
# print(model.cost_function_values)
plt.plot(model.cost_function_values,label="Cost Function")
plt.legend()
plt.show()
# print(model.test_model())
model.test_model()
plt.plot(model.metrics,label="Metrics")
plt.legend()
plt.show()
